In [1]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.tree import DecisionTreeClassifier

# -----------------------------------------------------------------------------
# 1. Transparent AdaBoost Engine (Step-by-Step State Tracker)
# -----------------------------------------------------------------------------
class AdaBoostTracker:
    """Trains AdaBoost.M1 and stores intermediate states for educational inspection."""

    def __init__(self, X, y, max_rounds=20):
        self.X = X
        # Map labels to {-1, +1} for classic AdaBoost arithmetic
        self.y = np.where(y == 0, -1, 1)
        self.N = len(X)
        self.max_rounds = max_rounds

        self.stumps = []
        self.alphas = []
        self.weighted_errors = []
        # weights_history[m] stores sample weights at the start of round m
        self.weights_history = [np.ones(self.N) / self.N]
        self.stump_preds = []
        self.cumulative_train_acc = []

        self._fit()

    def _fit(self):
        w = np.ones(self.N) / self.N
        cum_scores = np.zeros(self.N)

        for m in range(self.max_rounds):
            # 1. Fit weak learner (decision stump) using current sample weights
            stump = DecisionTreeClassifier(max_depth=1, random_state=42)
            stump.fit(self.X, self.y, sample_weight=w)
            pred = stump.predict(self.X)

            # 2. Weighted classification error
            misclassified = pred != self.y
            epsilon = np.sum(w[misclassified]) / np.sum(w)
            epsilon = np.clip(epsilon, 1e-10, 1 - 1e-10)

            # 3. Estimator weight (alpha)
            alpha = 0.5 * np.log((1.0 - epsilon) / epsilon)

            # 4. Update sample weights: w <- w * exp(-alpha * y * pred)
            w = w * np.exp(-alpha * self.y * pred)
            w = w / np.sum(w)  # Re-normalize

            # Record round variables
            self.stumps.append(stump)
            self.alphas.append(alpha)
            self.weighted_errors.append(epsilon)
            self.weights_history.append(w.copy())
            self.stump_preds.append(pred)

            # Running ensemble accuracy
            cum_scores += alpha * pred
            cum_acc = np.mean(np.sign(cum_scores) == self.y)
            self.cumulative_train_acc.append(cum_acc)


# -----------------------------------------------------------------------------
# 2. Data Synthesis Function
# -----------------------------------------------------------------------------
def get_dataset(name, n_samples=50, noise=0.2, seed=42):
    np.random.seed(seed)
    if name == "Moons":
        X, y = make_moons(n_samples=n_samples, noise=noise, random_state=seed)
    elif name == "Circles":
        X, y = make_circles(
            n_samples=n_samples, noise=noise, factor=0.5, random_state=seed
        )
    else:  # Two clusters with overlapping outliers
        X, y = make_blobs(
            n_samples=n_samples,
            centers=[[-1.2, -1.0], [1.2, 1.0]],
            cluster_std=1.0 + noise * 1.5,
            random_state=seed,
        )
    return X, y


# -----------------------------------------------------------------------------
# 3. Interactive UI & Plotting Dashboard
# -----------------------------------------------------------------------------
# Global domain bounds
X_LIM = (-3.2, 3.2)
Y_LIM = (-3.2, 3.2)
CMAP_BG = ListedColormap(["#ffebee", "#e3f2fd"])

# Widget Controls
dataset_dd = widgets.Dropdown(
    options=["Moons", "Circles", "Overlapping Blobs"],
    value="Moons",
    description="Dataset:",
    layout=widgets.Layout(width="220px"),
)

samples_slider = widgets.IntSlider(
    value=50,
    min=20,
    max=100,
    step=10,
    description="Samples (N):",
    layout=widgets.Layout(width="240px"),
)

noise_slider = widgets.FloatSlider(
    value=0.20,
    min=0.05,
    max=0.50,
    step=0.05,
    description="Noise:",
    layout=widgets.Layout(width="220px"),
)

round_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=15,
    step=1,
    description="Round (m):",
    layout=widgets.Layout(width="260px"),
)

btn_step_prev = widgets.Button(
    description="◀ Prev", layout=widgets.Layout(width="80px")
)
btn_step_next = widgets.Button(
    description="Next ▶",
    button_style="primary",
    layout=widgets.Layout(width="80px"),
)
btn_new_seed = widgets.Button(
    description="Resample", icon="refresh", layout=widgets.Layout(width="110px")
)

chk_weight_sizes = widgets.Checkbox(
    value=True,
    description="Scale points by weight $w_i$",
    indent=False,
    layout=widgets.Layout(width="230px"),
)
chk_highlight_errors = widgets.Checkbox(
    value=True,
    description="Highlight round misclassifications",
    indent=False,
    layout=widgets.Layout(width="260px"),
)

output_plot = widgets.Output()
output_stats = widgets.Output()

# Application state
state = {"seed": 42, "model": None, "grid_X": None}

# Precompute grid mesh for boundaries
gx = np.linspace(X_LIM[0], X_LIM[1], 150)
gy = np.linspace(Y_LIM[0], Y_LIM[1], 150)
XX, YY = np.meshgrid(gx, gy)
grid_points = np.c_[XX.ravel(), YY.ravel()]


def retrain_model():
    X, y = get_dataset(
        dataset_dd.value,
        n_samples=samples_slider.value,
        noise=noise_slider.value,
        seed=state["seed"],
    )
    state["model"] = AdaBoostTracker(
        X, y, max_rounds=round_slider.max
    )


def render():
    m_idx = round_slider.value - 1  # 0-indexed round
    model = state["model"]

    current_stump = model.stumps[m_idx]
    current_alpha = model.alphas[m_idx]
    current_eps = model.weighted_errors[m_idx]
    current_weights = model.weights_history[m_idx]
    current_pred = model.stump_preds[m_idx]
    misclassified = current_pred != model.y

    # Compute cumulative ensemble prediction up to round m
    cum_grid = np.zeros(grid_points.shape[0])
    for k in range(m_idx + 1):
        cum_grid += model.alphas[k] * model.stumps[k].predict(grid_points)
    cum_decision_mesh = np.sign(cum_grid).reshape(XX.shape)

    # Compute weak learner boundary on grid
    stump_decision_mesh = current_stump.predict(grid_points).reshape(XX.shape)

    with output_plot:
        output_plot.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), sharey=True)

        for ax in axes:
            ax.set_xlim(X_LIM)
            ax.set_ylim(Y_LIM)
            ax.grid(True, linestyle=":", alpha=0.5)

        # Marker sizes: scale base size (40) by normalized sample weight
        if chk_weight_sizes.value:
            # Multiply by N so an average-weighted sample has size ~45
            sizes = 45 * (current_weights * model.N)
            sizes = np.clip(sizes, 10, 450)
        else:
            sizes = np.full(model.N, 45)

        color_map = np.where(model.y == 1, "#1976d2", "#d32f2f")

        # -------------------------------------------------------------
        # Left Plot: Current Decision Stump m
        # -------------------------------------------------------------
        axes[0].contourf(
            XX,
            YY,
            stump_decision_mesh,
            levels=[-1.5, 0, 1.5],
            cmap=CMAP_BG,
            alpha=0.6,
        )
        axes[0].contour(
            XX,
            YY,
            stump_decision_mesh,
            levels=[0],
            colors="#212121",
            linewidths=2.0,
        )

        axes[0].scatter(
            model.X[:, 0],
            model.X[:, 1],
            s=sizes,
            c=color_map,
            edgecolors="black",
            linewidth=0.8,
            zorder=4,
        )

        if chk_highlight_errors.value and np.any(misclassified):
            axes[0].scatter(
                model.X[misclassified, 0],
                model.X[misclassified, 1],
                s=sizes[misclassified] + 80,
                facecolors="none",
                edgecolors="#ffd600",
                linewidth=2.5,
                label="Wrong by Stump",
                zorder=5,
            )
            axes[0].legend(loc="upper right", framealpha=0.9)

        axes[0].set_title(
            f"Round {m_idx + 1} Weak Learner $h_{{{m_idx + 1}}}(x)$ (Decision"
            " Stump)\nMarker size reflects training weight $w_i$",
            fontsize=11,
            fontweight="bold",
        )
        axes[0].set_xlabel("$x_1$")
        axes[0].set_ylabel("$x_2$")

        # -------------------------------------------------------------
        # Right Plot: Cumulative Ensemble H_m(x)
        # -------------------------------------------------------------
        axes[1].contourf(
            XX,
            YY,
            cum_decision_mesh,
            levels=[-1.5, 0, 1.5],
            cmap=CMAP_BG,
            alpha=0.6,
        )
        axes[1].contour(
            XX,
            YY,
            cum_decision_mesh,
            levels=[0],
            colors="#1b5e20",
            linewidths=2.4,
        )

        axes[1].scatter(
            model.X[:, 0],
            model.X[:, 1],
            s=45,
            c=color_map,
            edgecolors="black",
            linewidth=0.8,
            zorder=4,
        )

        axes[1].set_title(
            f"Ensemble $H_{{{m_idx + 1}}}(x) = "
            r"\mathrm{sign}\left(\sum_{k=1}^{"
            f"{m_idx + 1}"
            r"}\alpha_k h_k(x)\right)$"
            f"\nTrain Accuracy: {model.cumulative_train_acc[m_idx]:.1%}",
            fontsize=11,
            fontweight="bold",
        )
        axes[1].set_xlabel("$x_1$")

        plt.tight_layout()
        plt.show()

    # Dynamic Math & Metrics readout
    with output_stats:
        output_stats.clear_output(wait=True)
        html_card = f"""
        <div style="background:#f8f9fa; border:1px solid #dee2e6; border-radius:6px; padding:12px 18px; margin-top:8px; font-family:sans-serif; font-size:13px; line-height:1.6;">
            <div style="display:flex; justify-content:space-between; flex-wrap:wrap; gap:12px;">
                <div>
                    <b>Stump Weighted Error ($\epsilon_{{{m_idx+1}}}$):</b> 
                    <span style="color:#c62828; font-weight:bold;">{current_eps:.4f}</span>
                    <span style="color:#757575;">(Misclassified: {np.sum(misclassified)} / {model.N} points)</span>
                </div>
                <div>
                    <b>Stump Weight Vote ($\alpha_{{{m_idx+1}}}$):</b> 
                    <span style="color:#2e7d32; font-weight:bold;">{current_alpha:.4f}</span>
                    <span style="color:#757575;">[Formula: $\\frac{{1}}{{2}}\\ln\\frac{{1-\\epsilon}}{{\\epsilon}}$]</span>
                </div>
                <div>
                    <b>Overall Ensemble Accuracy:</b> 
                    <span style="color:#1565c0; font-weight:bold;">{model.cumulative_train_acc[m_idx]:.1%}</span>
                </div>
            </div>
            <div style="margin-top:8px; color:#424242;">
                💡 <b>Teaching Insight:</b> Notice how the misclassified points inside the yellow rings receive higher weights $w_i$, making their bubbles larger. In Round {min(m_idx+2, round_slider.max)}, the next stump is heavily penalized if it does not separate them.
            </div>
        </div>
        """
        display(HTML(html_card))


# -----------------------------------------------------------------------------
# 4. Callback Connections
# -----------------------------------------------------------------------------
def on_dataset_or_noise_change(change):
    retrain_model()
    render()


def on_round_change(change):
    render()


def on_step_next_clicked(b):
    if round_slider.value < round_slider.max:
        round_slider.value += 1


def on_step_prev_clicked(b):
    if round_slider.value > round_slider.min:
        round_slider.value -= 1


def on_resample_clicked(b):
    state["seed"] = np.random.randint(1, 10000)
    retrain_model()
    render()


dataset_dd.observe(on_dataset_or_noise_change, names="value")
samples_slider.observe(on_dataset_or_noise_change, names="value")
noise_slider.observe(on_dataset_or_noise_change, names="value")
round_slider.observe(on_round_change, names="value")
chk_weight_sizes.observe(lambda c: render(), names="value")
chk_highlight_errors.observe(lambda c: render(), names="value")

btn_step_next.on_click(on_step_next_clicked)
btn_step_prev.on_click(on_step_prev_clicked)
btn_new_seed.on_click(on_resample_clicked)

# -----------------------------------------------------------------------------
# 5. Dashboard Layout Assembly
# -----------------------------------------------------------------------------
row1 = widgets.HBox(
    [dataset_dd, samples_slider, noise_slider, btn_new_seed],
    layout=widgets.Layout(align_items="center", gap="12px"),
)
row2 = widgets.HBox(
    [btn_step_prev, btn_step_next, round_slider, chk_weight_sizes, chk_highlight_errors],
    layout=widgets.Layout(align_items="center", gap="12px", margin="8px 0"),
)

display(widgets.VBox([row1, row2, output_stats, output_plot]))

# Initial setup execution
retrain_model()
render()